In [ ]:
import tensorflow_datasets as tfds
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
(ds_train, ds_test), ds_info = tfds.load(
    "fashion_mnist",
    split = ["train", "test"],
    as_supervised = True,
    with_info = True,
    data_dir = "/path/to/thesis_project/data/raw/fashion_mnist")

In [ ]:
#dataset basic info

print(ds_info, "\n")
print(type(ds_info), "\n")

class_names = ds_info.features["label"].names
num_classes = ds_info.features["label"].num_classes

print("Numbber of training samples:", ds_info.splits["train"].num_examples)
print("Number of test samples:", ds_info.splits["test"].num_examples)
print("Number of classes:", num_classes)
print("Class names:", class_names)

for image, label in ds_train.take(1): #inspecting data format quickly
    print("Image shape:", image.shape)
    print("Image dtype:", image.dtype)
    print("Pixel range: [{}, {}]".format( int(image.numpy().min()), int(image.numpy().max()) ) )

In [ ]:
# pixel range check
pixel_min = np.inf # start with very large number
pixel_max = -np.inf # start with very small number

for image, _ in ds_train:
    img = image.numpy()
    pixel_min = min(pixel_min, img.min())
    pixel_max = max(pixel_max, img.max())

for image, _ in ds_test:
    img = image.numpy()
    pixel_min = min(pixel_min, img.min())
    pixel_max = max(pixel_max, img.max())

print("dtype:", ds_info.features["image"].dtype)
print("pixel min:", int(pixel_min))
print("pixel max:", int(pixel_max))

In [ ]:
#class distribution - samples per class (train)
counts = np.zeros(num_classes, dtype = int)
for _, label in ds_train:
    counts[int(label)] += 1

#plot
plt.figure(figsize=(10,4))
plt.bar(class_names, counts)
plt.title("fashion_mnist class distribution (train)")
plt.xlabel("Class")
plt.ylabel("Number of samples")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig("/path/to/thesis_project/eda/fashion_mnist_class_distribution_(train).png",
            dpi = 300, bbox_inches = "tight")
plt.show()

for name, count in zip(class_names, counts):
    print("{}: {}".format(name, count))

In [ ]:
#sample images per class
n_per_class = 5
samples = {i: [] for i in range(num_classes)}
for image, label in ds_train:
    l = int(label)
    if len(samples[l]) < n_per_class:
        samples[l].append(image.numpy())
    if all(len(v) == n_per_class for v in samples.values()):
        break

#plot
fig, axes = plt.subplots(num_classes, n_per_class, figsize=(10, 20))
for i in range(num_classes):
    for j in range(n_per_class):
        axes[i, j].imshow(samples[i][j].squeeze(), cmap="gray")
        axes[i, j].axis("off")
plt.tight_layout()
plt.savefig("/path/to/thesis_project/eda/fashion_mnist_sample_images_per_class.png",
            dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
#pixel statistics

pixel_sum = 0.0
pixel_sq_sum = 0.0
n_pixels = 0

for image, _ in ds_train:
    img = image.numpy().astype(np.float32) / 255.0
    pixel_sum += img.sum()
    pixel_sq_sum += (img**2).sum()
    n_pixels += img.size

mean = pixel_sum / n_pixels
std = np.sqrt( (pixel_sq_sum / n_pixels) - (mean**2) )

print("pixel mean: {:.4f}".format(mean))
print("pixel std: {:.4f}".format(std))

#plot
all_pixels = []
for image, _ in ds_train:
    all_pixels.append((image.numpy().flatten()) / 255.0)
all_pixels = np.concatenate(all_pixels)

plt.figure(figsize=(10,4))
plt.hist(all_pixels)
plt.title("fashion_mnist pixel statistics (train)")
plt.xlabel("pixel value")
plt.ylabel("frequency")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig("/path/to/thesis_project/eda/fashion_mnist_pixel_statistics_(train).png",
            dpi = 300, bbox_inches = "tight")
plt.show()